## tl;dr

- Срез: пуши отдельного приложения Blizko, отправленные в июле 2026 года.
- Покупки: только заказы, фактически оформленные в отдельном приложении Blizko.
- Атрибуция: последний клик по пушу, окно 24 часа.
- На момент среза: 13 пушей, 109 заказов, 90 уникальных покупателей и 112 428 ₽ выручки.

## Context & Methods

### Key Assumptions

- Локальный дашборд запущен на `http://localhost:3000`.
- `/api/reports/blizko-july` читает данные из Supabase без локальных демо-данных.
- CTR считается от доставленных сообщений, конверсия — от открытий в заказ.
- Исследовательские рассылки отделены от коммерческих и не участвуют в ранжировании продаж.

## Data

### 1. Load the reviewed report payload

In [ ]:
import json
from urllib.request import urlopen

REPORT_URL = "http://localhost:3000/api/reports/blizko-july"
with urlopen(REPORT_URL) as response:
    report = json.load(response)

report["summary"]

## Results

### 2. Recompute the headline metrics and ranking

In [ ]:
campaigns = report["campaigns"]
commercial = [row for row in campaigns if row["type"] == "commercial"]

recomputed = {
    "campaigns": len(campaigns),
    "sent": sum(row["sent"] for row in campaigns),
    "delivered": sum(row["delivered"] for row in campaigns),
    "clicked": sum(row["clicked"] for row in campaigns),
    "orders": sum(row["orders"] for row in campaigns),
    "revenue": sum(row["revenue"] for row in campaigns),
}
assert all(recomputed[key] == report["summary"][key] for key in recomputed)

ranking = sorted(
    (
        {
            "title": row["title"],
            "orders": row["orders"],
            "ctr": row["clicked"] / max(row["delivered"], 1),
            "click_to_order": row["orders"] / max(row["clicked"], 1),
            "revenue": row["revenue"],
        }
        for row in commercial
    ),
    key=lambda row: row["orders"],
    reverse=True,
)
ranking[:3]

## Takeaways

- `Любимый вкус ✨` лидирует по числу заказов: 27.
- `ЧМ уже в разгаре ⚽` лидирует по конверсии клика в заказ: 26,25% и по выручке: 25 017 ₽.
- Два лидера дали 48 из 109 заказов. Их взвешенный CTR — 1,48%, конверсия после клика — 22,12%.
- У остальных коммерческих пушей взвешенный CTR — 0,85%, конверсия после клика — 15,17%.
- Это наблюдательная атрибуция, поэтому объяснения влияния текста нужно подтвердить A/B-тестом.